# Lecția 10 - Agenți AI în Producție

În această lecție vei învăța **modele de producție** pentru agenți AI folosind Microsoft Agent Framework (Python). Acoperim:

- **Observabilitate** — măsurarea timpului și jurnalizarea interacțiunilor agentului
- **Evaluare** — folosirea unui agent evaluator pentru a nota calitatea răspunsurilor
- **Gestionarea costurilor** — strategii pentru optimizarea tokenilor și selecția modelelor

Scenariul este un **agent de călătorii** care ajută utilizatorii să planifice călătorii, cu monitorizare și evaluare suprapuse.


## Configurare


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Considerații privind producția

Mutarea agenților AI din prototipuri în producție necesită o atenție deosebită la trei piloni:

1. **Observabilitate** — Ai nevoie de vizibilitate asupra a ceea ce face agentul, cât durează și ce instrumente apelează. Fără urmărire și înregistrare, depanarea problemelor din producție este aproape imposibilă.

2. **Evaluare** — Verificările automate de calitate asigură că răspunsurile agentului rămân precise, complete și utile în timp. Un agent de evaluare poate puncta răspunsurile în funcție de criterii definite.

3. **Gestionarea costurilor** — Utilizarea tokenilor afectează direct costul. Strategii precum optimizarea promptului, selectarea modelului și stocarea în cache ajută la menținerea cheltuielilor sub control fără a sacrifica calitatea.


## Construirea unui agent observabil

Definim instrumentele de călătorie și învelim apelul agentului cu măsurare a timpului astfel încât să putem monitoriza latența. În producție, ai integra OpenTelemetry sau un backend de tracing similar.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Tipare de evaluare

Un tipar comun în producție este utilizarea unui al doilea agent ca **evaluător**. Evaluătorul evaluează răspunsul agentului principal în raport cu criterii predefinite, cum ar fi completitudinea, acuratețea și utilitatea.

Acest lucru permite:
- Porți automate de control al calității înainte ca răspunsurile să ajungă la utilizatori
- Detectarea regresiei atunci când prompturile sau modelele se schimbă
- Monitorizare continuă a performanței agentului în timp


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategii de gestionare a costurilor

Controlul costurilor este esențial pentru agenții AI din producție. Iată strategiile principale:

| Strategie | Descriere |
|---|---|
| **Optimizarea prompturilor** | Menține instrucțiunile sistemului concise. Elimină contextul redundant pentru a reduce tokeni de intrare. |
| **Selectarea modelului** | Folosește modele mai mici și mai ieftine (de ex. GPT-4o-mini) pentru sarcini simple precum clasificare sau extragere și rezervă modelele mai mari pentru raționamente complexe. |
| **Stocare în cache** | Salvează în cache rezultatele instrumentelor și interogările frecvente pentru a evita apelurile API redundante. |
| **Bugete de tokeni** | Setează limite pentru `max_tokens` pentru a preveni răspunsuri neașteptat de lungi. |
| **Procesare în loturi** | Grupează mai multe interogări ale utilizatorilor într-un singur apel API, atunci când este posibil. |

În practică, o abordare pe niveluri funcționează bine: direcționează cererile simple către un model rapid și ieftin și escaladează doar interogările complexe către unul mai capabil.


## Rezumat

În această lecție ai învățat cum să:

1. **Adăuga observabilitate** la interacțiunile agenților prin măsurarea timpului și jurnalizare, punând bazele pentru trasare și monitorizare.
2. **Evalua răspunsurile agenților** automat folosind un agent evaluator care acordă scoruri pentru completitudine, acuratețe și utilitate.
3. **Gestiona costurile** prin optimizarea prompturilor, selecția modelului, stocarea în cache și bugete de tokeni.

Aceste modele de producție contribuie la asigurarea faptului că agenții tăi AI sunt fiabili, măsurabili și rentabili la scară.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Declinare de responsabilitate:
Acest document a fost tradus folosind serviciul de traducere cu inteligență artificială Co-op Translator (https://github.com/Azure/co-op-translator). Deși ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original, în limba sa nativă, trebuie considerat sursa oficială. Pentru informații critice, se recomandă o traducere profesională realizată de un traducător uman. Nu ne asumăm răspunderea pentru eventualele neînțelegeri sau interpretări greșite care pot rezulta din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
